In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import matplotlib.pyplot as plt
import torch

from cyclegan_core import denormalize, seed_everything
from two_stage_virtual_staining import (
    ColorizerTrainer, build_structure_trainer, build_two_stage_dataloaders
)

# Two-stage virtual H&E staining at 2.0 MPP

Stage 1 uses a paired bidirectional CycleGAN to complete `Unstain OD ↔ H&E grayscale OD`. Stage 2 colorizes predicted H&E OD into RGB H&E. The complete 2048×2048 patch at 0.5 MPP is resized to 512×512 at 2.0 MPP.

In [ ]:
data_params = {
    'seed': 42,
    'gpu_index': 1,
    'data_dir': Path('../../data/HnE_n_UNStaining/patch_dataset_mpp05_2048'),
    'image_ext': 'png',
    'image_max_count': 30000,
    'original_size': 2048,
    'source_mpp': 0.5,
    'target_mpp': 2.0,
    'input_size': 512,
    'batch_size': 2,
    'val_fraction': 0.10,
    'preload_images': True,
    'max_cache_gib': 64,
    'od_background_threshold': 0.98,
    'od_quantile': 0.995,
    'od_calibration_images': 256,
}

structure_params = {
    **data_params,
    'output_dir': Path('../../results/Unstain2HnE_two_stage/structure'),
    'checkpoint_dir': Path('../../model/Unstain2HnE_two_stage/structure'),
    'paired_training': True,
    'domain_b_mode': 'od',
    'num_epochs': 100,
    'decay_start_epoch': 50,
    'ngf': 32,
    'ndf': 64,
    'residual_blocks': 6,
    'lr': 2e-4,
    'beta1': 0.5,
    'beta2': 0.999,
    'lambda_cycle': 10.0,
    'lambda_identity': 5.0,
    'lambda_background': 5.0,
    'lambda_paired_blur': 5.0,
    'paired_blur_kernel': 5,
    'paired_blur_sigma': 0.8,
    'a_background_od_threshold': 0.04,
    'b_background_od_threshold': 0.04,
    'background_mask_blur_kernel': 5,
    'pool_size': 50,
    'preview_count': 2,
    'save_every': 10,
    'resume_checkpoint': None,
}

color_params = {
    'output_dir': Path('../../results/Unstain2HnE_two_stage/color'),
    'checkpoint_dir': Path('../../model/Unstain2HnE_two_stage/color'),
    'num_epochs': 100,
    'base_channels': 32,
    'ndf': 64,
    'lr_g': 2e-4,
    'lr_d': 1e-4,
    'beta1': 0.5,
    'beta2': 0.999,
    'lambda_gan': 1.0,
    'lambda_rgb': 20.0,
    'lambda_ssim': 2.0,
    'lambda_background': 10.0,
    'background_od_threshold': 0.04,
    'background_mask_blur_kernel': 5,
    'predicted_od_start_epoch': 10,
    'predicted_od_ramp_epochs': 20,
    'max_predicted_od_probability': 0.50,
    'preview_count': 2,
    'save_every': 10,
}

seed_everything(data_params['seed'])
if torch.cuda.is_available():
    device = torch.device(f"cuda:{data_params['gpu_index']}")
    torch.backends.cudnn.benchmark = True
else:
    device = torch.device('cpu')
print('device:', device)

## DataLoader

Unstain and H&E use the same filename, full field of view, and spatial augmentation. Separate fixed global OD maxima are calibrated at the actual 512×512 training scale; no per-image min-max or z-score normalization is used.

In [ ]:
data = build_two_stage_dataloaders(data_params)
print('Unstain OD_MAX:', data['unstain_od_max'])
print('H&E OD_MAX:', data['hne_od_max'])

In [ ]:
unstain_od, hne_od, hne_rgb = next(iter(data['color_train']))
columns = min(4, len(unstain_od))
fig, axes = plt.subplots(3, columns, figsize=(4 * columns, 10), squeeze=False)
for i in range(columns):
    axes[0, i].imshow(denormalize(unstain_od[i]).permute(1, 2, 0).numpy(), cmap='gray')
    axes[0, i].set_title('Unstain OD')
    axes[1, i].imshow(denormalize(hne_od[i]).permute(1, 2, 0).numpy(), cmap='gray')
    axes[1, i].set_title('Target H&E OD structure')
    axes[2, i].imshow(denormalize(hne_rgb[i]).permute(1, 2, 0).numpy())
    axes[2, i].set_title('Target RGB H&E')
    for row in range(3):
        axes[row, i].axis('off')
plt.tight_layout()

## Stage 1 — paired CycleGAN structure completion

Both directions remain active: `Unstain OD → H&E OD → Unstain OD` and `H&E OD → Unstain OD → H&E OD`. A mild 5×5, sigma 0.8 blurred paired loss supplements cycle, identity, adversarial, and OD-background losses.

In [ ]:
structure_trainer = build_structure_trainer(structure_params, data, device)

In [ ]:
structure_trainer.fit()

In [ ]:
best_structure_path = structure_params['checkpoint_dir'] / 'best.pt'
best_structure = torch.load(
    best_structure_path, map_location=device, weights_only=False
)
structure_trainer.G_AB.load_state_dict(best_structure['G_AB'])
structure_trainer.G_BA.load_state_dict(best_structure['G_BA'])
structure_trainer.G_AB.eval()
print('Loaded best structure epoch:', best_structure['epoch'] + 1)

## Stage 2 — H&E OD to RGB H&E colorization

The colorizer first learns from exact `Real H&E OD → Real RGB H&E` pairs. Starting at color epoch 11, Stage-1 predicted OD is mixed in gradually up to 50%, preventing a train/inference input-distribution gap. Stage-1 generators remain frozen.

In [ ]:
color_trainer = ColorizerTrainer(
    color_params, data['color_train'], data['color_val'],
    structure_trainer.G_AB, device
)

In [ ]:
color_trainer.fit()

## Full-pipeline validation

`predicted_*` metrics evaluate the full `Unstain OD → predicted H&E OD → RGB H&E` pipeline. `oracle_*` metrics use real H&E OD and measure the upper bound of the colorization stage alone.

In [ ]:
metrics, preview = color_trainer.validate()
print(metrics)
color_trainer.save_preview(max(color_trainer.start_epoch - 1, 0), preview)